# 04 — Modelagem: Regressão de Inadimplência PJ (t+6)

Treina e avalia **5 modelos de ML** para prever a taxa de inadimplência real daqui a 6 meses,
comparados a referenciais ingênuos — persistência (`ŷ(t+6)` = último valor publicado) e
sazonais (t−6, t−12) — e ao ARIMA(1,1,1) com origem móvel por segmento.

**Pergunta central**: os modelos de ML superam baselines simples (persistência e ARIMA) o
suficiente para justificar sua complexidade?

## Modelos e baselines
| Modelo | Tipo | Justificativa |
|--------|------|---------------|
| Persistência (t+6=t) | Baseline ingênuo | Referência mínima: "repete o último valor" |
| ARIMA(1,1,1) | Baseline estatístico | Referência clássica de série temporal |
| Regressão Linear | Linear | Baseline ML interpretável |
| Árvore de Decisão | Não-linear | Regras de decisão explícitas |
| Random Forest | Ensemble | Reduz variância via bagging |
| XGBoost | Ensemble (boosting) | Estado da arte em dados tabulares |
| MLP | Rede neural | Captura não-linearidades complexas |

## Métricas
- **RMSE** — penaliza erros grandes (importante nas viradas de ciclo)
- **MAE** — erro absoluto médio (interpretação direta em p.p.)
- **R²** — variância explicada; reportado *pooled* e *within-segment* (ver §5.1)
- **MASE** — erro absoluto escalado pelo naive de 1 passo (padrão em forecasting); **MASE < 1** = melhor que o ingênuo. Substitui o MAPE, inútil aqui pois o alvo é ≈0 em muitos segmentos.

## Validação temporal (sem leakage)
- Teste: 2022-01 → 2024-06; o treino termina `embargo_meses` antes do teste (lido de `ml_metadata.json`).
- **Embargo = horizonte + defasagem de publicação do SCR** (6 + 2 = 8 meses): só entram no treino
  linhas cujo alvo (t+6) já estaria publicado na data da previsão.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

BASE = Path().resolve().parent
PROC = BASE / 'data' / 'processed'
MDIR = PROC / 'models'
MDIR.mkdir(parents=True, exist_ok=True)

with open(PROC / 'ml_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

FEATURE_COLS = meta['feature_cols']
TARGET_COL   = meta['target_col']
SPLIT_DATE   = pd.Timestamp(meta['split_date'])

print('Pacotes carregados')
print(f'Features: {len(FEATURE_COLS)}  |  Target: {TARGET_COL}')
print(f'Split: treino < {SPLIT_DATE.date()}  |  teste >= {SPLIT_DATE.date()}')

## 1. Carga e split temporal

In [ ]:
# IMPORTANTE: ordena por data para que TimeSeriesSplit produza folds temporais validos.
# (O ml_dataset.parquet vem ordenado por SEGMENTO; sem este sort a CV separa grupos de
#  segmentos, nao passado/futuro, e a validacao cruzada temporal fica invalida.)
df = (
    pd.read_parquet(PROC / 'ml_dataset.parquet')
    .sort_values('data_base')
    .reset_index(drop=True)
)

# --- EMBARGO temporal no split ---
# O alvo de uma linha de treino em d e a taxa em d+H. Essa taxa so e PUBLICADA em d+H+LAG_SCR
# (revisao A1: defasagem de publicacao do SCR). Para que todo alvo de treino ja fosse conhecido
# na data da previsao, removem-se do treino os ultimos H+LAG_SCR meses antes do split.
HORIZONTE = int(meta.get('horizonte', 6))
LAG_SCR   = int(meta.get('lag_scr', 0))
EMBARGO   = int(meta.get('embargo_meses', HORIZONTE))
# Nivel conhecido na origem: ultimo valor PUBLICADO (nivel_disp) quando ha defasagem
NIVEL     = meta.get('nivel_col', 'taxa_inadimplencia') if LAG_SCR else 'taxa_inadimplencia'
embargo_lim = SPLIT_DATE - pd.DateOffset(months=EMBARGO)

df_train = df[df['data_base'] < embargo_lim].copy()   # treino com embargo (sem leakage no alvo)
df_test  = df[df['data_base'] >= SPLIT_DATE].copy()
n_embargo = int(((df['data_base'] >= embargo_lim) & (df['data_base'] < SPLIT_DATE)).sum())

X_train_raw = df_train[FEATURE_COLS]
X_test_raw  = df_test[FEATURE_COLS]
y_train = df_train[TARGET_COL].values
y_test  = df_test[TARGET_COL].values

print(f'Defasagem de publicacao do SCR: {LAG_SCR} mes(es)  |  nivel conhecido na origem: {NIVEL}')
print(f'Embargo de {EMBARGO} meses (horizonte {HORIZONTE} + defasagem {LAG_SCR}): treino vai ate {embargo_lim.date()} '
      f'(removidas {n_embargo:,} linhas de {embargo_lim.date()}..{(SPLIT_DATE - pd.DateOffset(days=1)).date()})')
print(f'Treino: {len(df_train):,} linhas  ({df_train["data_base"].min().date()} -> {df_train["data_base"].max().date()})')
print(f'Teste:  {len(df_test):,}  linhas  ({df_test["data_base"].min().date()} -> {df_test["data_base"].max().date()})')
print(f'\nTarget (treino) — media: {y_train.mean():.4f}  std: {y_train.std():.4f}')
print(f'Target (teste)  — media: {y_test.mean():.4f}  std: {y_test.std():.4f}')

## 2. Baseline ARIMA(1,1,1) — origem móvel (rolling)

Para cada segmento (cnae_secao × porte × modalidade) ajustamos um ARIMA(1,1,1) e geramos
previsões com **origem móvel**: para cada mês de origem `t` do período de teste, o modelo é
condicionado na informação disponível **até `t`** e prevê `t+6` — exatamente o mesmo alvo que
os modelos de ML. A atualização entre origens usa `append(refit=False)` (filtro de Kalman, sem
reestimar), o que torna a comparação **justa e alinhada linha-a-linha** ao conjunto de teste de ML.

> Correção em relação à versão anterior, que fazia um único `forecast` em bloco a partir do fim
> do treino e avaliava sobre um conjunto de pontos diferente — comparação enviesada contra o ML.

O ARIMA serve como **baseline estatístico clássico**: se os modelos de ML não superarem o ARIMA
(e o baseline ingênuo de persistência), não justificam sua complexidade adicional.

**ARIMA(p=1, d=1, q=1)**: `d=1` (diferenciação remove tendência), `p=1` (1 termo AR), `q=1` (1 termo MA).

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

SEG_KEYS = ['cnae_secao', 'porte', 'modalidade']


def arima_rolling_forecast(df_all, split_date, seg_keys, horizonte=6, lag_pub=0,
                           col='taxa_inadimplencia'):
    """ARIMA(1,1,1) com origem movel, alinhado linha-a-linha ao alvo t+h.

    Para cada origem t, condiciona o modelo na informacao PUBLICADA ate t e preve t+h. Com
    `lag_pub` > 0 (defasagem de publicacao, revisao A1), na origem r o modelo so conhece a serie
    ate r - lag_pub e precisa prever horizonte + lag_pub passos a frente para alcancar t+h.
    Com lag_pub = 0 a funcao e identica a versao original.
    O resultado e gravado na LINHA da origem (onde o alvo de ML, target = taxa(t+h), esta
    definido), garantindo comparacao na MESMA amostra do ML. Usa append(refit=False).
    """
    pred = pd.Series(np.nan, index=df_all.index, dtype=float)
    split = np.datetime64(pd.Timestamp(split_date))
    passos = horizonte + lag_pub
    n_ok = n_fail = 0

    for _, grp in df_all.groupby(seg_keys, observed=True):
        grp = grp.sort_values('data_base')
        y = grp[col].to_numpy(dtype=float)
        rows = grp.index.to_numpy()
        dates = grp['data_base'].to_numpy()
        n = len(y)
        n_train = int((dates < split).sum())
        k = n_train - lag_pub                      # observacoes conhecidas na primeira origem

        if k < 12 or n_train >= n:
            n_fail += 1
            continue
        try:
            ext = ARIMA(y[:k], order=(1, 1, 1)).fit()
        except Exception:
            n_fail += 1
            continue

        ok = False
        for r in range(n_train - 1, n):            # r = origem da previsao
            try:
                while k < r - lag_pub + 1:         # incorpora o que foi publicado ate a origem
                    ext = ext.append([y[k]], refit=False)
                    k += 1
                if r + horizonte < n:              # existe alvo y[r+h]
                    pred.loc[rows[r]] = ext.forecast(steps=passos)[passos - 1]
                    ok = True
            except Exception:
                break
        n_ok += int(ok)

    print(f'Segmentos OK: {n_ok}  |  Falhas: {n_fail}')
    return pred


print(f'Rodando ARIMA(1,1,1) rolling por segmento (defasagem de publicacao = {LAG_SCR}; '
      f'preve {HORIZONTE + LAG_SCR} passos)...')
pred_arima = arima_rolling_forecast(df, SPLIT_DATE, SEG_KEYS, horizonte=HORIZONTE, lag_pub=LAG_SCR)

# Alinha ARIMA EXATAMENTE as linhas de teste de ML (mesmo alvo t+6, mesmas linhas)
mask_test    = df['data_base'] >= SPLIT_DATE
y_true_arima = df.loc[mask_test, 'target'].to_numpy()
y_pred_arima = pred_arima.loc[mask_test].to_numpy()
ok = ~np.isnan(y_pred_arima) & ~np.isnan(y_true_arima)
y_true_arima, y_pred_arima = y_true_arima[ok], y_pred_arima[ok]
print(f'Pontos comparados (ARIMA alinhado ao teste de ML): {ok.sum():,} de {mask_test.sum():,}')

# Metricas ARIMA (na mesma amostra do ML)
rmse_arima = np.sqrt(mean_squared_error(y_true_arima, y_pred_arima))
mae_arima  = mean_absolute_error(y_true_arima, y_pred_arima)
r2_arima   = r2_score(y_true_arima, y_pred_arima)
mape_arima = np.nan  # MAPE removido: alvo ~0 em muitos segmentos torna a metrica inutil

print(f'\nARIMA(1,1,1) rolling — metricas:')
print(f'  RMSE:  {rmse_arima:.4f}')
print(f'  MAE:   {mae_arima:.4f}')
print(f'  R²:    {r2_arima:.4f}')

## 3. Modelos de Machine Learning

Todos os modelos seguem o mesmo padrão:
1. Pipeline com `SimpleImputer(median)` → `StandardScaler` (onde aplicável) → regressor
2. Validação cruzada temporal (`TimeSeriesSplit`) no conjunto de treino
3. Avaliação final no conjunto de teste (2022-2024)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Revisao A4a: para o modelo linear, as categoricas (codificadas como inteiros arbitrarios)
# entram em one-hot e as numericas sao padronizadas. A versao original (label encoding
# tratado como numero) e mantida para comparacao.
CAT_COLS = [c for c in FEATURE_COLS if c.endswith('_enc')]
NUM_COLS = [c for c in FEATURE_COLS if c not in CAT_COLS]

def pre_onehot():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('scl', StandardScaler())]), NUM_COLS),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_COLS),
    ])

# Pipelines
pipelines = {
    'LinearRegression': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('reg', LinearRegression()),
    ]),
    'LinearRegression (one-hot)': Pipeline([
        ('pre', pre_onehot()),
        ('reg', LinearRegression()),
    ]),
    'DecisionTree': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('reg', DecisionTreeRegressor(max_depth=8, min_samples_leaf=20, random_state=42)),
    ]),
    'RandomForest': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('reg', RandomForestRegressor(
            n_estimators=200, max_depth=12,
            min_samples_leaf=10, n_jobs=-1, random_state=42
        )),
    ]),
    'XGBoost': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('reg', XGBRegressor(
            n_estimators=300, learning_rate=0.05,
            max_depth=6, subsample=0.8,
            colsample_bytree=0.8, random_state=42,
            verbosity=0, eval_metric='rmse',
        )),
    ]),
    'MLP': Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', StandardScaler()),
        ('reg', MLPRegressor(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu', max_iter=300,
            early_stopping=True, validation_fraction=0.1,
            random_state=42,
        )),
    ]),
}

print(f'Categoricas (one-hot na variante A4a): {CAT_COLS}')
print('Pipelines definidos:')
for name, pipe in pipelines.items():
    print(f'  {name}: {" -> ".join(pipe.named_steps.keys())}')

In [ ]:
# Validacao cruzada temporal — TimeSeriesSplit no treino
tscv = TimeSeriesSplit(n_splits=5)

cv_results = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(
        pipe, X_train_raw, y_train,
        cv=tscv, scoring='neg_root_mean_squared_error',
        n_jobs=-1,
    )
    rmse_cv = -scores
    cv_results[name] = {'mean': rmse_cv.mean(), 'std': rmse_cv.std()}
    print(f'{name:20s}  RMSE CV: {rmse_cv.mean():.4f} ± {rmse_cv.std():.4f}')

print('\nValidacao cruzada concluida')

## 4. Avaliação no conjunto de teste

In [ ]:
results = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train_raw, y_train)
    y_pred = pipe.predict(X_test_raw)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / np.clip(np.abs(y_test), 0.001, None))) * 100

    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'MAPE': mape, 'y_pred': y_pred}

    # Salva pipeline
    fname = (name.lower().replace(' (one-hot)', '_onehot')
             .replace('linearregression','lr').replace('decisiontree','dt')
             .replace('randomforest','rf').replace('xgboost','xgb').replace('mlp','mlp'))
    joblib.dump(pipe, MDIR / f'{fname}.pkl')
    print(f'{name:28s}  RMSE:{rmse:.4f}  MAE:{mae:.4f}  R²:{r2:.4f}  MAPE:{mape:.1f}%')

# Adiciona ARIMA
results['ARIMA(1,1,1)'] = {
    'RMSE': rmse_arima, 'MAE': mae_arima, 'R2': r2_arima, 'MAPE': mape_arima,
    'y_pred': y_pred_arima
}

print('\nModelos salvos em:', MDIR)

In [ ]:
# === Baselines INGENUOS (persistencia e sazonais) + metricas MASE ===
# Persistencia: "a inadimplencia daqui a 6 meses = a ultima CONHECIDA na origem".
# Com defasagem de publicacao (revisao A1), o ultimo valor conhecido e nivel_disp = taxa(t-LAG).
y_persist = df_test[NIVEL].to_numpy(dtype=float)

rmse_p = np.sqrt(mean_squared_error(y_test, y_persist))
mae_p  = mean_absolute_error(y_test, y_persist)
r2_p   = r2_score(y_test, y_persist)
results['Persistencia (t+6=t)'] = {
    'RMSE': rmse_p, 'MAE': mae_p, 'R2': r2_p, 'MAPE': np.nan, 'y_pred': y_persist,
}
print(f'Persistencia (ultimo valor publicado = {NIVEL})  RMSE:{rmse_p:.4f}  MAE:{mae_p:.4f}  R²:{r2_p:.4f}')

# --- Revisao A4b: referenciais SAZONAIS ingenuos: y(t+6) = taxa(t-6) e y(t+6) = taxa(t-12) ---
for rot, col in [('Sazonal (t+6=t-6)', 'taxa_m6'), ('Sazonal (t+6=t-12)', 'taxa_m12')]:
    if col not in df_test.columns:
        print(f'  {rot}: coluna {col} ausente, pulado')
        continue
    ys = df_test[col].to_numpy(dtype=float)
    oks = np.isfinite(ys)
    results[rot] = {
        'RMSE': np.sqrt(mean_squared_error(y_test[oks], ys[oks])),
        'MAE':  mean_absolute_error(y_test[oks], ys[oks]),
        'R2':   r2_score(y_test[oks], ys[oks]),
        'MAPE': np.nan, 'y_pred': ys, 'n': int(oks.sum()),
    }
    print(f'{rot:28s} RMSE:{results[rot]["RMSE"]:.4f}  MAE:{results[rot]["MAE"]:.4f}  '
          f'R²:{results[rot]["R2"]:.4f}  (n={oks.sum():,} de {len(ys):,})')

# --- MASE (Mean Absolute Scaled Error, Hyndman 2006) ---
# Escala = MAE do naive IN-SAMPLE (no treino), por segmento.
#   MASE  : naive de 1 passo (padrao da literatura; nao e o horizonte avaliado)
#   MASE6 : naive de 6 passos (revisao A4d) — a escala coerente com o horizonte t+6:
#           MASE6 < 1 significa melhor que repetir o valor de 6 meses atras, no treino.
g_tr = (df_train.sort_values(SEG_KEYS + ['data_base'])
                .groupby(SEG_KEYS, observed=True)['taxa_inadimplencia'])
scale_mase  = g_tr.diff(1).abs().mean()
scale_mase6 = g_tr.diff(HORIZONTE).abs().mean()
print(f'\nEscala MASE  (MAE naive 1-passo, in-sample): {scale_mase:.5f}')
print(f'Escala MASE6 (MAE naive {HORIZONTE}-passos, in-sample): {scale_mase6:.5f}\n')

for name, d in results.items():
    d['MASE']  = d['MAE'] / scale_mase
    d['MASE6'] = d['MAE'] / scale_mase6
    print(f'  {name:28s} MASE: {d["MASE"]:.3f}   MASE6: {d["MASE6"]:.3f}')

print('\nMASE < 1 -> melhor que o naive de 1 passo; MASE6 < 1 -> melhor que o naive de 6 passos (ambos in-sample).')

## 5. Tabela comparativa de resultados

In [ ]:
COLS = ['RMSE', 'MAE', 'R2', 'MASE', 'MASE6']   # MAPE removido (inutil: alvo ~0); MASE/MASE6 no lugar

df_results = pd.DataFrame({
    k: {m: v for m, v in v.items() if m in COLS}
    for k, v in results.items()
}).T[COLS].round(4)

df_results = df_results.sort_values('RMSE')
df_results.index.name = 'Modelo'

print('=== Resultados no conjunto de teste (2022-2024) ===')
print(df_results.to_string())

# Salva CSV
df_results.to_csv(PROC / 'resultados_modelos.csv')
print('\nSalvo: resultados_modelos.csv')

### 5.1 Decomposição: R² *pooled* vs *within-segment*

O R² da tabela acima é **pooled** (global): ele recompensa o modelo por distinguir segmentos
estruturalmente mais inadimplentes dos menos — algo que os *lags* entregam quase de graça, pois
cada segmento "carrega" seu próprio nível.

O **R² within-segment** mede a habilidade *real* de prever a variação **temporal dentro de cada
segmento**, comparando o erro contra a média do próprio segmento no teste (baseline within). Se
`R²_within ≪ R²_pooled`, boa parte da "variância explicada" vinha de diferenças **entre**
segmentos, não de previsão temporal — uma leitura honesta do que o modelo de fato aprende.

In [ ]:
# === R² e RMSE WITHIN-SEGMENT — duas referencias ===
# Identificador de segmento alinhado as linhas de teste (mesma ordem de y_test / y_pred do ML).
seg_test = df_test[SEG_KEYS].reset_index(drop=True)
gid = seg_test.astype(str).agg('|'.join, axis=1).to_numpy()
yv  = y_test

# Referencia 1 (ORACULO): media do proprio segmento NO TESTE. Nao e um preditor operacional —
# serve apenas para DECOMPOR a variancia (quanto do R² global vem de diferencas entre segmentos).
seg_mean_te  = pd.Series(yv).groupby(gid).transform('mean').to_numpy()
ss_within_te = float(np.sum((yv - seg_mean_te) ** 2))

# Referencia 2 (HONESTA, revisao A4e): media do segmento NO TREINO — informacao disponivel na
# origem. Segmento sem historico no treino recebe a media global do treino.
gid_tr = df_train[SEG_KEYS].astype(str).agg('|'.join, axis=1).to_numpy()
m_tr   = pd.Series(y_train).groupby(gid_tr).mean()
seg_mean_tr  = pd.Series(gid).map(m_tr).fillna(float(y_train.mean())).to_numpy()
ss_within_tr = float(np.sum((yv - seg_mean_tr) ** 2))

ss_tot = float(np.sum((yv - yv.mean()) ** 2))
print(f'SST (teste) = {ss_tot:.4f} | SS dentro (oraculo, media no teste) = {ss_within_te:.4f} '
      f'({ss_within_te/ss_tot:.1%}) | SS vs media do segmento no treino = {ss_within_tr:.4f}')

rows = []
for name, d in results.items():
    if name == 'ARIMA(1,1,1)':       # y_pred em subamostra diferente — fora desta decomposicao
        continue
    yp = np.asarray(d['y_pred'], dtype=float)
    if len(yp) != len(yv) or not np.all(np.isfinite(yp)):
        continue
    ss_res       = float(np.sum((yv - yp) ** 2))
    r2_within_tr = 1 - ss_res / ss_within_tr
    r2_within_te = 1 - ss_res / ss_within_te
    rmse_seg     = float(np.sqrt(pd.Series((yv - yp) ** 2).groupby(gid).mean()).mean())
    rows.append((name, round(d['R2'], 4), round(r2_within_tr, 4), round(r2_within_te, 4), round(rmse_seg, 4)))

# Linha de controle: a propria media do segmento no treino como preditor
ss_ctrl = ss_within_tr
rows.append(('Media do segmento (treino)', round(r2_score(yv, seg_mean_tr), 4),
             0.0, round(1 - ss_ctrl / ss_within_te, 4),
             round(float(np.sqrt(pd.Series((yv - seg_mean_tr) ** 2).groupby(gid).mean()).mean()), 4)))

dfw = (
    pd.DataFrame(rows, columns=['Modelo', 'R2_pooled', 'R2_within_treino', 'R2_within', 'RMSE_seg_medio'])
    .set_index('Modelo')
    .sort_values('R2_within_treino', ascending=False)
)
print('\n=== R² global vs intrassegmento (R2_within_treino = vs media do segmento no TREINO; '
      'R2_within = vs media no TESTE, oraculo) ===')
print(dfw.to_string())
dfw.to_csv(PROC / 'resultados_within_segment.csv')
print('\nSalvo: resultados_within_segment.csv')
print('\n> R2_within_treino > 0: o modelo supera a media historica do segmento (referencia operacional).')
print('> R2_within (oraculo) << R2_pooled: parte do R² global vem de diferencas ENTRE segmentos.')

In [ ]:
# Grafico RMSE comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

modelos = df_results.index.tolist()
cores   = ['#e41a1c' if m == 'ARIMA(1,1,1)' else '#377eb8' for m in modelos]

# RMSE
axes[0].barh(modelos, df_results['RMSE'], color=cores)
axes[0].set_xlabel('RMSE')
axes[0].set_title('RMSE — quanto menor melhor')
axes[0].axvline(df_results.loc['ARIMA(1,1,1)', 'RMSE'], color='#e41a1c', linestyle='--', alpha=0.5, label='ARIMA baseline')
axes[0].legend(fontsize=8)
axes[0].grid(True, axis='x', alpha=0.3)

# R2
axes[1].barh(modelos, df_results['R2'], color=cores)
axes[1].set_xlabel('R²')
axes[1].set_title('R² — quanto maior melhor')
axes[1].axvline(df_results.loc['ARIMA(1,1,1)', 'R2'], color='#e41a1c', linestyle='--', alpha=0.5, label='ARIMA baseline')
axes[1].legend(fontsize=8)
axes[1].grid(True, axis='x', alpha=0.3)

plt.suptitle('Comparativo de modelos — conjunto de teste (2022-2024)', fontsize=13)
plt.tight_layout()
plt.savefig(PROC / 'resultados_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: resultados_comparativo.png')

## 6. Real vs. Previsto — melhores modelos

In [ ]:
# Top 3 ML + ARIMA  (exclui os baselines do ranking de ML)
_baselines = ['ARIMA(1,1,1)', 'Persistencia (t+6=t)', 'Sazonal (t+6=t-6)', 'Sazonal (t+6=t-12)']
top_ml = df_results[~df_results.index.isin(_baselines)].head(3).index.tolist()
plot_models = top_ml + ['ARIMA(1,1,1)']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, name in enumerate(plot_models):
    ax = axes[i]
    if name == 'ARIMA(1,1,1)':
        yt = y_true_arima
        yp = results[name]['y_pred']
    else:
        yt = y_test
        yp = results[name]['y_pred']

    ax.scatter(yt, yp, alpha=0.3, s=8, color='#377eb8')
    lim = max(yt.max(), yp.max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='y=x')
    ax.set_xlabel('Real')
    ax.set_ylabel('Previsto')
    r2  = results[name]['R2']
    rmse = results[name]['RMSE']
    ax.set_title(f'{name}\nRMSE={rmse:.4f}  R²={r2:.4f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Real vs. Previsto — taxa de inadimplência (t+6)', fontsize=13)
plt.tight_layout()
plt.savefig(PROC / 'resultados_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: resultados_scatter.png')

## 7. Distribuição dos resíduos

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, name in enumerate(plot_models):
    ax = axes[i]
    if name == 'ARIMA(1,1,1)':
        residuos = y_true_arima - results[name]['y_pred']
    else:
        residuos = y_test - results[name]['y_pred']

    ax.hist(residuos, bins=60, color='#4575b4', alpha=0.8, edgecolor='none')
    ax.axvline(0, color='red', linewidth=1.2, linestyle='--')
    ax.axvline(residuos.mean(), color='orange', linewidth=1, linestyle='-', label=f'media={residuos.mean():.4f}')
    ax.set_title(f'{name} — residuos')
    ax.set_xlabel('Real - Previsto')
    ax.set_ylabel('Frequencia')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

plt.suptitle('Distribuicao dos residuos — taxa de inadimplencia (t+6)', fontsize=13)
plt.tight_layout()
plt.savefig(PROC / 'resultados_residuos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: resultados_residuos.png')

In [ ]:
print('=== Resumo para comparativo academico ===')

baselines = ['ARIMA(1,1,1)', 'Persistencia (t+6=t)', 'Sazonal (t+6=t-6)', 'Sazonal (t+6=t-12)']
df_ml_only = df_results[~df_results.index.isin(baselines)]

best_ml      = df_ml_only.index[0]
best_ml_rmse = df_ml_only['RMSE'].min()
print(f'\nMelhor modelo de ML: {best_ml}  (RMSE={best_ml_rmse:.4f})')

# Ganho vs cada baseline (em reducao de RMSE)
for bl in baselines:
    if bl in df_results.index:
        bl_rmse = df_results.loc[bl, 'RMSE']
        ganho = (bl_rmse - best_ml_rmse) / bl_rmse * 100
        rel = f'{ganho:.1f}% menor RMSE que' if ganho > 0 else f'{-ganho:.1f}% MAIOR RMSE que'
        print(f'  {best_ml} tem {rel} {bl}  (RMSE {bl}={bl_rmse:.4f})')

print('\n> Leitura honesta: o ganho relevante e o do ML sobre a PERSISTENCIA, nao sobre o ARIMA.')
print('> Se o ML mal supera "repetir o ultimo valor", o sinal vem da autocorrelacao, nao do modelo.')

print(f'\nTabela completa:')
print(df_results[['RMSE', 'MAE', 'R2', 'MASE', 'MASE6']].to_string())

## 9. Teste de significância — Diebold-Mariano

O ganho do melhor modelo sobre os baselines poderia ser ruído amostral. O teste de
**Diebold-Mariano** (Diebold e Mariano, 1995) verifica se a diferença de acurácia preditiva
entre dois modelos é estatisticamente significativa.

Como o conjunto de teste é um **painel** (segmentos × 30 meses), o diferencial de perda é
agregado por mês (média entre segmentos) e o teste é aplicado à série mensal resultante, com
variância de longo prazo **Newey-West** (defasagem h−1 = 5, pois previsões t+6 com origens
sobrepostas são autocorrelacionadas) e a **correção de amostra pequena de
Harvey–Leybourne–Newbold** (1997).

- **H₀**: os dois modelos têm a mesma acurácia esperada.
- **Perda quadrática (SE)** alinha-se ao RMSE (sensível à cauda); **perda absoluta (AE)** alinha-se ao MAE (caso típico).
- Convenção: `d̄ > 0` com `p < 0,05` → o **segundo** modelo do par é significativamente melhor.

In [ ]:
from scipy import stats as sps

def dm_test_panel(dates, err_a, err_b, h=6, loss='se'):
    """Diebold-Mariano para painel: agrega o diferencial de perda por MES (media entre
    segmentos) e testa a serie mensal com variancia Newey-West (lag h-1) e correcao de
    amostra pequena de Harvey-Leybourne-Newbold (HLN).

    d = L(err_a) - L(err_b).  d_bar > 0 com p<0.05  =>  modelo B (segundo) e melhor."""
    if loss == 'se':
        d = err_a**2 - err_b**2
    else:
        d = np.abs(err_a) - np.abs(err_b)
    s = pd.Series(d).groupby(pd.Series(dates).values).mean().sort_index()
    T = len(s)
    dbar = float(s.mean())
    u = (s - dbar).to_numpy()
    lrv = float(u @ u) / T
    L = h - 1
    for lag in range(1, min(L, T - 1) + 1):
        w = 1 - lag / (L + 1)
        lrv += 2 * w * float(u[lag:] @ u[:-lag]) / T
    lrv = max(lrv, 1e-16)
    dm = dbar / np.sqrt(lrv / T)
    hln = np.sqrt(max((T + 1 - 2 * h + h * (h - 1) / T) / T, 1e-12))
    dm_hln = dm * hln
    p = 2 * (1 - sps.t.cdf(abs(dm_hln), df=T - 1))
    return {'T_meses': T, 'd_bar': dbar, 'DM_HLN': round(dm_hln, 3), 'p_valor': round(p, 4)}

dates_test = df_test['data_base'].values
err = {
    'RandomForest': y_test - np.asarray(results['RandomForest']['y_pred'], dtype=float),
    'XGBoost':      y_test - np.asarray(results['XGBoost']['y_pred'], dtype=float),
    'Persistencia': y_test - np.asarray(results['Persistencia (t+6=t)']['y_pred'], dtype=float),
}
# ARIMA alinhado linha-a-linha as observacoes de teste (NaN onde nao ha previsao)
pa = pred_arima.loc[df_test.index].to_numpy()
ok_a = ~np.isnan(pa)
err_arima = y_test[ok_a] - pa[ok_a]

testes = [
    ('Persistencia vs RandomForest', 'se', err['Persistencia'], err['RandomForest'], dates_test),
    ('Persistencia vs XGBoost',      'se', err['Persistencia'], err['XGBoost'],      dates_test),
    ('ARIMA vs RandomForest',        'se', err_arima, err['RandomForest'][ok_a],     dates_test[ok_a]),
    ('Persistencia vs RandomForest', 'ae', err['Persistencia'], err['RandomForest'], dates_test),
]
linhas = []
for nome, loss, ea, eb, dt in testes:
    r = dm_test_panel(dt, ea, eb, h=6, loss=loss)
    r['d_bar'] = f"{r['d_bar']:.2e}"
    linhas.append({'Par (A vs B)': nome, 'Perda': loss.upper(), **r})

df_dm = pd.DataFrame(linhas)
print('=== Diebold-Mariano (agregacao mensal, NW lag=5, correcao HLN) ===')
print('Convencao: d_bar > 0 e p < 0.05  =>  o modelo B (segundo do par) e significativamente melhor\n')
print(df_dm.to_string(index=False))
df_dm.to_csv(PROC / 'resultados_diebold_mariano.csv', index=False)
print('\nSalvo: resultados_diebold_mariano.csv')

## 10. Robustez por sub-janelas (2022 · 2023 · 2024)

Avalia a **estabilidade temporal** do ranking dividindo o teste em três sub-períodos.
Se o melhor modelo trocasse a cada ano, o resultado global seria específico do período;
se o ranking se mantém, a conclusão é robusta a regimes distintos (aperto monetário em
2022–2023, acomodação em 2024). Observação: 2024 cobre apenas janeiro–junho (limite de
disponibilidade do alvo t+6).

In [ ]:
# === Robustez por sub-janela: RMSE por ano do teste ===
anos = sorted(df_test['data_base'].dt.year.unique())
year_arr = df_test['data_base'].dt.year.to_numpy()

rows = []
for nome, d in results.items():
    if nome == 'ARIMA(1,1,1)':
        yp_full = pred_arima.loc[df_test.index].to_numpy()
    else:
        yp_full = np.asarray(d['y_pred'], dtype=float)
    valid = np.isfinite(yp_full)
    linha = {'Modelo': nome}
    for a in anos:
        m = (year_arr == a) & valid
        if m.sum() == 0:
            continue
        linha[f'RMSE {a}'] = round(float(np.sqrt(mean_squared_error(y_test[m], yp_full[m]))), 4)
    rows.append(linha)

df_sub = pd.DataFrame(rows).set_index('Modelo')
df_sub = df_sub.sort_values(f'RMSE {anos[0]}')
print('=== RMSE por sub-janela do teste (2024 = jan-jun) ===')
print(df_sub.to_string())

print('\nMelhor modelo por ano:')
for c in df_sub.columns:
    print(f'  {c[-4:]}: {df_sub[c].idxmin():28s} (RMSE {df_sub[c].min():.4f})')

df_sub.to_csv(PROC / 'resultados_subjanelas.csv')
print('\nSalvo: resultados_subjanelas.csv')